# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leiandrei/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

For my chosen lane and provided question, I am gonna use a regression method since, I am using a numerical values as my target variables to determine whether those specific characteristics affect the observable content or page features for engagements.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# import libs
import pandas as pd
import numpy as np

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target variable would be the `engagement_rate` from the dataframe, and a regression task would be appropriate since I will be dealing with a continuous value. The variable is an observed outcome from different users browsing a certain content or page.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.00,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.00,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.00,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.00,good,page_3_5,down,-34.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_c322796023c8,client_e29c9c180c,10.0,0.05,LOW,0.00,keyword article,transactional,1386.0,9084.0,...,8000-15000,0.00,0.0,0.00,0.00,0.00,low,top_3,new,NaN
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,15000-25000,0.39,6.6,0.00,66.67,0.00,moderate,page_1,down,-75.1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,15000-25000,0.19,4.1,0.00,0.00,0.00,good,page_1,down,-66.2
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,NaN,0.22,6.0,1.73,4.06,0.00,excellent,page_1,down,-27.9


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Using an 80% threshold, we define a "highly engaging" page as one that sits in the top 20% of historical engagement (the 80th percentile). Precision@50 measures what fraction of the top 50 pages recommended by the model actually meet or exceed this 80th percentile threshold.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
clean_df = df[df['sessions_90d'] >= 50]
clean_df = clean_df.dropna(subset=['engagement_rate'])

threshold = clean_df['engagement_rate'].quantile(.80)

clean_df['target'] = clean_df['engagement_rate'] >= threshold

baseline_ranking = clean_df.sort_values(by='sessions_90d', ascending=False)

k = 50
top_k_baseline = baseline_ranking.head(k)

correct_predictions = top_k_baseline['target'].sum()
precision_at_k = correct_predictions / k

print('Baseline Evaluation')
print(f"Engagement Threshold Target at: {threshold:.1f}")
print(f"Baseline Precision@K: {precision_at_k:.3f}")
print(f"the model is considered 'good' if Precision@50 > {precision_at_k:.3f}")

Baseline Evaluation
Engagement Threshold Target at: 4.4
Baseline Precision@K: 0.260
the model is considered 'good' if Precision@50 > 0.260


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one unique pseudonymized content item (a specific webpage) aggregated over a fixed 90-day historical window. Each row pairs the observable structural characteristics of that page (e.g., word count, content age, format) with its actual user engagement outcomes over that exact same period.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
analysis_cols = [
    'content_id', 'client_id', 'word_count', 
    'content_age_days', 'content_type', 
    'sessions_90d', 'engagement_rate'
]

available_cols = [col for col in analysis_cols if col in clean_df.columns]
unit_df = clean_df[available_cols].copy()

unit_df.head()

,content_id,client_id,word_count,content_age_days,content_type,sessions_90d,engagement_rate
3,content_331d6c4de07b,client_19581e27de,NaN,463,keyword article,78,1.28
4,content_d99b7a2d90ca,client_3fdba35f04,2803.0,263,keyword article,145,0.00
8,content_5e6c160719bc,client_6208ef0f77,3807.0,90,keyword article,68,5.88
10,content_d8ee6cc6d642,client_19581e27de,NaN,329,keyword article,326,6.75
12,content_42fb2cad9ecf,client_6208ef0f77,3969.0,124,keyword article,233,3.43


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

User engagement usually do not follow a linear relationship, or a single variable unit, foe example, using "if" a certain number is greater than the threshold is highly engaging, it can be misleading since we are looking upon different factors or characteristics that also drives a highly engagement rates. ML models such a regressors perfectly capture the non linear interactions.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.